# OpenAI Agent Stack — Practical Jupyter Notebook

This notebook turns the lesson on the **OpenAI Agent Stack** into a practical Python workflow.

We will explore:

1. What an AI agent is.
2. The OpenAI Agent Stack.
3. OpenAI models.
4. The Responses API.
5. Tool calling.
6. Building a simple manual agent loop.
7. The OpenAI Agents SDK.
8. Multi-step agent workflows.
9. A practical flight-search example.
10. Responses API vs Agents SDK.
11. When to choose each approach.

## Core Mental Model

An AI agent can be understood as:

**Agent = LLM + Tools + Loop**

- **LLM** → The brain that reasons.
- **Tools** → The hands that perform actions.
- **Loop** → The orchestration mechanism that repeatedly reasons, acts, and observes.

The goal is to move from simple LLM calls toward systems that can dynamically choose actions and complete multi-step tasks.

# 1. The OpenAI Agent Stack

A simplified view of the stack is:

```text
┌─────────────────────────────────────┐
│          Your Application           │
│ Business Logic • UI • Custom Code   │
└─────────────────────────────────────┘
                  │
┌─────────────────────────────────────┐
│          OpenAI Agents SDK           │
│ Agents • Handoffs • Guardrails      │
│ Orchestration • Tracing             │
└─────────────────────────────────────┘
                  │
┌─────────────────────────────────────┐
│           Responses API              │
│ Tools • Function Calling • State    │
└─────────────────────────────────────┘
                  │
┌─────────────────────────────────────┐
│         OpenAI Models                │
│ Reasoning and Non-Reasoning Models  │
└─────────────────────────────────────┘
```

The important relationship is:

**Agents SDK → Responses API → OpenAI Models**

You can use the Responses API directly when you need more control, or use the Agents SDK for higher-level agent orchestration.

# 2. AI Agent Mental Model

A traditional LLM application often looks like:

```text
User
  ↓
LLM
  ↓
Answer
```

An agentic application looks more like:

```text
User
  ↓
LLM reasons
  ↓
Choose a tool
  ↓
Execute tool
  ↓
Observe result
  ↓
Reason again
  ↓
Choose another tool
  ↓
Final answer
```

This is the **Reason → Act → Observe** loop.

The agent can repeat the loop until it determines that the task is complete.

# 3. Install the OpenAI Python SDK

Install or upgrade the OpenAI Python SDK.

The Agents SDK is installed separately.

In [ ]:
%pip install -U openai openai-agents python-dotenv

# 4. Configure the OpenAI API Key

Create a `.env` file:

```text
OPENAI_API_KEY=your_openai_api_key
```

Never commit API keys to source code or Git repositories.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is missing. "
        "Add OPENAI_API_KEY to your .env file."
    )

print("OpenAI API key found.")

# 5. Import the OpenAI Client

In [ ]:
from openai import OpenAI

client = OpenAI()

print("OpenAI client initialized.")

# 6. Simple Responses API Call

The Responses API can be used directly without the Agents SDK.

This is useful when:

- The workflow is simple.
- You want direct control.
- You do not need complex multi-agent orchestration.
- You want to manage the application loop yourself.

In [ ]:
response = client.responses.create(
    model="gpt-5.5",
    input="Explain what an AI agent is in three sentences."
)

print(response.output_text)

# 7. Understanding the Responses API

The Responses API can work with:

- Text generation
- Tool calling
- Function calling
- Built-in tools
- Conversation state
- Multi-turn interactions

The API is the lower-level building block for creating agentic applications.

For a simple question, the flow may be:

```text
User
 ↓
Responses API
 ↓
Model
 ↓
Final Answer
```

For tool use:

```text
User
 ↓
Responses API
 ↓
Model requests tool
 ↓
Your application executes tool
 ↓
Tool result sent back
 ↓
Model continues reasoning
 ↓
Final answer
```

# 8. Create a Simple Tool

We will create a flight-search tool.

For this notebook, the function is a **mock implementation**.
It does not call a real airline or travel API.

The purpose is to demonstrate the agent architecture.

In [ ]:
def search_flights(origin: str, destination: str, date: str):
    """Search available flights between two airports for a specific date."""

    return {
        "origin": origin,
        "destination": destination,
        "date": date,
        "flights": [
            {
                "airline": "Example Airways",
                "flight": "EA101",
                "price": 620,
                "currency": "USD",
                "duration": "10h 30m",
            },
            {
                "airline": "Demo Airlines",
                "flight": "DA202",
                "price": 540,
                "currency": "USD",
                "duration": "12h 10m",
            },
            {
                "airline": "Sample Air",
                "flight": "SA303",
                "price": 710,
                "currency": "USD",
                "duration": "9h 45m",
            },
        ],
    }

search_flights(
    origin="AUS",
    destination="ZRH",
    date="2026-08-15",
)

# 9. Define the Tool Schema

When using function calling, the model needs to know:

- Tool name
- Tool description
- Input parameters
- Parameter types
- Required fields

This schema allows the model to decide when and how to call the function.

In [ ]:
import json

flight_tool = {
    "type": "function",
    "name": "search_flights",
    "description": (
        "Search available flights between an origin and destination "
        "airport for a specific date."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "origin": {
                "type": "string",
                "description": "Origin airport code, such as AUS.",
            },
            "destination": {
                "type": "string",
                "description": "Destination airport code, such as ZRH.",
            },
            "date": {
                "type": "string",
                "description": "Travel date in YYYY-MM-DD format.",
            },
        },
        "required": ["origin", "destination", "date"],
        "additionalProperties": False,
    },
}

print(json.dumps(flight_tool, indent=2))

# 10. Responses API with a Tool

Now we provide the model with a tool.

The model can decide whether it needs to call the tool.

In [ ]:
response = client.responses.create(
    model="gpt-5.5",
    input=(
        "Find me the cheapest flight from Austin to Zurich "
        "on 2026-08-15."
    ),
    tools=[flight_tool],
)

for item in response.output:
    print(item)

# 11. Inspect Tool Calls

The model may return a function call instead of a final natural-language answer.

Conceptually:

```text
Model
  ↓
I need flight information.
  ↓
Call search_flights
  ↓
Arguments:
  origin = AUS
  destination = ZRH
  date = 2026-08-15
```

Your application must execute the function and send its result back to the model.

In [ ]:
tool_calls = [
    item
    for item in response.output
    if getattr(item, "type", None) == "function_call"
]

for call in tool_calls:
    print("Tool:", call.name)
    print("Arguments:", call.arguments)
    print("Call ID:", call.call_id)

# 12. Manual Agent Loop

This is the important concept from the lesson.

When using the Responses API directly, your application can manage the loop.

The simplified flow is:

1. Send user input and tools to the model.
2. Check whether the model requested a tool.
3. Parse the arguments.
4. Execute the Python function.
5. Add the tool result to the conversation.
6. Call the model again.
7. Repeat until there are no more tool calls.

In [ ]:
def run_manual_agent(user_question: str):
    response = client.responses.create(
        model="gpt-5.5",
        input=user_question,
        tools=[flight_tool],
    )

    while True:
        tool_calls = [
            item
            for item in response.output
            if getattr(item, "type", None) == "function_call"
        ]

        if not tool_calls:
            return response.output_text

        tool_outputs = []

        for call in tool_calls:
            if call.name == "search_flights":
                arguments = json.loads(call.arguments)

                result = search_flights(
                    origin=arguments["origin"],
                    destination=arguments["destination"],
                    date=arguments["date"],
                )

                tool_outputs.append(
                    {
                        "type": "function_call_output",
                        "call_id": call.call_id,
                        "output": json.dumps(result),
                    }
                )

        response = client.responses.create(
            model="gpt-5.5",
            previous_response_id=response.id,
            input=tool_outputs,
            tools=[flight_tool],
        )

# 13. Run the Manual Agent

The application manages the orchestration loop itself.

This approach provides maximum control, but as the application grows, the loop can become more complex.

For example, you may need to add:

- Retry handling
- Multiple tools
- Tool errors
- Timeouts
- Guardrails
- Human approval
- Multiple agents
- Agent handoffs

In [ ]:
answer = run_manual_agent(
    "Find me the cheapest flight from Austin to Zurich on 2026-08-15."
)

print(answer)

# 14. The Manual Loop in Detail

The architecture is:

```text
User
 ↓
Your Python Code
 ↓
Responses API
 ↓
LLM
 ↓
Tool Call
 ↓
Your Python Code
 ↓
Execute search_flights()
 ↓
Tool Result
 ↓
Responses API
 ↓
LLM
 ↓
Final Answer
```

This is powerful, but your application owns the orchestration logic.

This is the main reason higher-level frameworks such as the OpenAI Agents SDK can be useful.

# 15. Install and Import the Agents SDK

The Agents SDK provides a higher-level abstraction for building agent applications.

In [ ]:
from agents import Agent, Runner, function_tool

print("OpenAI Agents SDK imported.")

# 16. Create an Agents SDK Tool

The decorator exposes the Python function as a tool.

The SDK can use the function signature and documentation to understand the tool.

In [ ]:
@function_tool
def flight_search_tool(
    origin: str,
    destination: str,
    date: str,
) -> str:
    """Search flights between two airports for a given date."""

    result = search_flights(
        origin=origin,
        destination=destination,
        date=date,
    )

    return json.dumps(result)

# 17. Create an Agent

The agent contains:

- Instructions
- Model configuration
- Tools

The SDK handles much of the orchestration for us.

In [ ]:
flight_agent = Agent(
    name="Flight Search Agent",
    instructions=(
        "You are a helpful travel assistant. "
        "Use the flight search tool when the user asks "
        "about available flights. "
        "When comparing flights, identify the cheapest option."
    ),
    tools=[flight_search_tool],
)

print(flight_agent.name)

# 18. Run the Agents SDK Agent

The user asks a natural-language question.

The SDK manages the agent execution flow.

In [ ]:
result = await Runner.run(
    flight_agent,
    "Find me the cheapest flight from Austin to Zurich on 2026-08-15."
)

print(result.final_output)

# 19. Multi-Step Agent Example

Now consider a more complex request:

> Find the cheapest flight from Austin to Zurich and book it.

This requires multiple steps:

1. Search flights.
2. Compare prices.
3. Select the cheapest flight.
4. Ask for confirmation or authorization.
5. Book the selected flight.
6. Handle booking errors.
7. Return confirmation.

A simple LLM call is not enough.

The application needs an agentic loop and potentially multiple tools.

# 20. Create a Mock Booking Tool

For demonstration, we will create a mock booking function.

This function does not make a real booking.

In [ ]:
@function_tool
def book_flight(
    flight_number: str,
    origin: str,
    destination: str,
    date: str,
) -> str:
    """Book a selected flight. This is a mock implementation."""

    return json.dumps(
        {
            "status": "confirmed",
            "booking_reference": "DEMO-12345",
            "flight_number": flight_number,
            "origin": origin,
            "destination": destination,
            "date": date,
        }
    )

# 21. Create a Multi-Step Flight Agent

The agent now has two tools:

- Flight search
- Flight booking

The agent can reason about the sequence.

In [ ]:
booking_agent = Agent(
    name="Flight Booking Agent",
    instructions=(
        "You are a travel booking assistant. "
        "First search for flights when needed. "
        "Compare the available options and identify the cheapest flight. "
        "Only book a flight when the user explicitly asks you to book it. "
        "Never claim a booking succeeded unless the booking tool returns success."
    ),
    tools=[
        flight_search_tool,
        book_flight,
    ],
)

print(booking_agent.name)

# 22. Run the Multi-Step Agent

The agent may perform:

```text
User Request
    ↓
Search Flights
    ↓
Observe Results
    ↓
Choose Cheapest Flight
    ↓
Book Flight
    ↓
Observe Booking Result
    ↓
Final Response
```

The exact execution path is dynamically determined by the agent.

In [ ]:
result = await Runner.run(
    booking_agent,
    (
        "Find the cheapest flight from Austin to Zurich "
        "on 2026-08-15 and book it."
    ),
)

print(result.final_output)

# 23. Responses API vs Agents SDK

| Capability | Responses API | Agents SDK |
|---|---|---|
| Direct model interaction | Yes | Yes |
| Tool calling | Yes | Yes |
| Manual orchestration | Yes | Possible |
| Agent loop abstraction | You manage it | SDK manages it |
| Multi-agent workflows | Manual implementation | Built-in abstractions |
| Guardrails | Application-managed | SDK support |
| Handoffs | Manual | SDK support |
| Tracing | Lower-level/custom | Built-in tooling |
| Control | Maximum | Higher-level |
| Development speed | More code | Faster for complex agents |

The important point is not that one is always better.

The choice depends on your application's complexity and the amount of control you need.

# 24. Decision Tree

A practical decision process:

```text
Do you need multi-step agentic logic?
        │
        ├── No
        │    ↓
        │  Use Responses API
        │
        └── Yes
             ↓
       Do you need multiple agents,
       guardrails, handoffs, or
       complex orchestration?
             │
             ├── No
             │    ↓
             │  Responses API is still possible,
             │  but you manage the loop yourself.
             │
             └── Yes
                  ↓
             Consider Agents SDK
```

### Simple mental model

**Responses API** → More direct control.

**Agents SDK** → Higher-level agent orchestration.

# 25. Example: Simple Task

User:

> What is the weather in London?

Possible architecture:

```text
User
 ↓
Responses API
 ↓
Weather Tool
 ↓
Result
 ↓
Final Answer
```

If only one tool call is required, the Responses API may be sufficient.

# 26. Example: Complex Task

User:

> Find me the cheapest flight from Austin to Zurich,
> check whether I need a visa,
> compare hotels near the airport,
> and book everything.

Possible architecture:

```text
User
 ↓
Travel Agent
 ├── Flight Search Agent
 │     └── Flight Tool
 │
 ├── Visa Agent
 │     └── Web/Search Tool
 │
 ├── Hotel Agent
 │     └── Hotel Search Tool
 │
 └── Booking Agent
       └── Booking Tool
```

This is where a higher-level agent framework becomes increasingly valuable.

# 27. Important Architecture Insight

The OpenAI Agents SDK does not replace the underlying model.

It provides an orchestration layer around model interactions and tools.

The conceptual stack remains:

```text
Application
    ↓
Agents SDK
    ↓
Responses API
    ↓
OpenAI Model
```

The agent still relies on the model for reasoning.

Tools still perform real-world actions.

The SDK helps coordinate the overall process.

# 28. Agent Properties

A well-designed agent is generally:

### Goal-directed
It works toward a defined objective.

### Autonomous
It can dynamically choose the next action.

### Tool-using
It can interact with external systems.

### Iterative
It can repeat the reasoning and action cycle.

### Observable
Its behavior should ideally be traceable and debuggable.

### Controlled
It should operate within defined permissions and guardrails.

# 29. Practical Design Guidelines

When building an agent application:

1. Start with the simplest architecture.
2. Use the Responses API for straightforward workflows.
3. Add tools only when the model needs external capabilities.
4. Introduce an agent framework when orchestration becomes complex.
5. Add guardrails around sensitive operations.
6. Require human approval for high-impact actions.
7. Log and trace tool calls.
8. Validate tool inputs.
9. Handle failures and retries.
10. Never let the model directly perform uncontrolled side effects.

# 30. Final Takeaways

The key ideas from this lesson are:

### AI Agent

**LLM + Tools + Loop**

### OpenAI Agent Stack

**Models → Responses API → Agents SDK → Application**

### Responses API

Use it when you want:

- Direct control
- Simple workflows
- Custom orchestration
- Manual control over the agent loop

### Agents SDK

Use it when you need:

- Multi-step agents
- Multiple agents
- Handoffs
- Guardrails
- Higher-level orchestration
- Easier agent development

### The Core Difference

With the **Responses API**, you can build and control the loop yourself.

With the **Agents SDK**, the framework provides higher-level abstractions for managing agent workflows.

The best choice depends on the complexity of your application.

# 31. Practical Exercise

Try modifying this notebook to build a travel agent that can:

1. Search flights.
2. Search hotels.
3. Compare prices.
4. Ask the user for confirmation.
5. Book a flight.
6. Book a hotel.
7. Return a complete travel itinerary.

Then consider:

- Which tools should be separate?
- Should there be one agent or multiple agents?
- Where should human approval happen?
- What happens if flight booking succeeds but hotel booking fails?
- How would you add retries?
- How would you trace each tool call?

These questions move the example from a simple demo toward real-world agent architecture.